<a href="https://colab.research.google.com/github/maulanasarowis/CodeIgniter-Ion-Auth/blob/master/broadcastTelegramMultiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import csv
import os
import time
import requests
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output
from google.colab import userdata

# ============================================================
# === MOUNT GOOGLE DRIVE (jalankan sekali di awal) ===========
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# === KONFIGURASI DASAR ===
BOT_TOKEN = userdata.get('BOT_TOKEN')

# Semua file disimpan di Google Drive agar tidak hilang saat runtime reset
DRIVE_DIR   = '/content/drive/MyDrive/TelegramBroadcast'
CSV_FILE    = os.path.join(DRIVE_DIR, 'id_telegram_test.csv')
LOG_FILE    = os.path.join(DRIVE_DIR, 'broadcast_log.csv')
IMAGE_DIR   = os.path.join(DRIVE_DIR, 'images')

# Buat folder jika belum ada
os.makedirs(DRIVE_DIR,  exist_ok=True)
os.makedirs(IMAGE_DIR,  exist_ok=True)

print(f"✅ Google Drive terhubung.")
print(f"📁 Folder kerja : {DRIVE_DIR}")
print(f"📋 CSV karyawan : {CSV_FILE}")
print(f"📝 File log     : {LOG_FILE}")
print(f"🖼️  Folder gambar: {IMAGE_DIR}")

# ============================================================
# === WIDGET INPUT ===========================================
# ============================================================
mode_selector = widgets.RadioButtons(
    options=['📝 Hanya Teks', '🖼️ Hanya Gambar + Caption', '📝🖼️ Teks + Gambar'],
    value='📝 Hanya Teks',
    description='Mode:',
    disabled=False
)

text_message = widgets.Textarea(
    placeholder='Tulis pesan broadcast di sini...',
    description='Pesan:',
    layout=widgets.Layout(width='100%', height='120px')
)

upload_button = widgets.FileUpload(
    accept='image/*',
    multiple=True,
    description='📤 Upload Gambar',
    layout=widgets.Layout(display='none')
)

captions_container = widgets.VBox([])

send_button = widgets.Button(
    description='🚀 Kirim Broadcast',
    button_style='success',
    icon='send'
)

reset_log_button = widgets.Button(
    description='🗑️ Reset Log (mulai ulang)',
    button_style='danger',
)

status_label = widgets.HTML(value="")
output_area  = widgets.Output()

# ============================================================
# === KONFIGURASI RATE LIMIT =================================
# ============================================================
DELAY_ANTAR_PESAN  = 0.5   # detik antar user (aman untuk 30msg/dtk)
DELAY_ANTAR_GAMBAR = 0.3   # detik antar gambar dalam 1 user
DELAY_RETRY        = 2.0   # detik sebelum retry
MAX_RETRY          = 3     # jumlah percobaan ulang
MAX_CAPTION        = 1024  # batas karakter caption Telegram


# ============================================================
# === FUNGSI KIRIM TEKS ======================================
# ============================================================
def send_message_to_chat(chat_id, text, retry=MAX_RETRY):
    url = f'https://api.telegram.org/bot{BOT_TOKEN}/sendMessage'
    for attempt in range(retry):
        try:
            resp = requests.post(url, data={
                'chat_id': chat_id, 'text': text, 'parse_mode': 'HTML'
            }, timeout=10)

            if resp.status_code == 200:
                return 'success'

            # Tangani 429: Telegram kasih tahu harus tunggu berapa detik
            if resp.status_code == 429:
                retry_after = resp.json().get('parameters', {}).get('retry_after', 5)
                with output_area:
                    print(f"⏳ Rate limited. Tunggu {retry_after} detik...")
                time.sleep(retry_after + 1)
                continue

            # Error lain (misal chat tidak ditemukan, bot diblokir)
            err_desc = resp.json().get('description', '')
            if 'blocked' in err_desc or 'not found' in err_desc or 'deactivated' in err_desc:
                return f'failed:{err_desc}'  # tidak perlu retry

            time.sleep(DELAY_RETRY)

        except requests.exceptions.Timeout:
            time.sleep(DELAY_RETRY)
        except Exception as e:
            time.sleep(DELAY_RETRY)

    return 'failed'


# ============================================================
# === FUNGSI KIRIM FOTO ======================================
# ============================================================
def send_photo_to_chat(chat_id, image_path, caption, retry=MAX_RETRY):
    """
    Kirim foto.
    - Jika caption <= 1024 karakter: kirim bersama foto (normal).
    - Jika caption > 1024 karakter : foto dikirim tanpa caption,
      lalu teks panjang dikirim sebagai pesan terpisah setelahnya.
    """
    url_photo = f'https://api.telegram.org/bot{BOT_TOKEN}/sendPhoto'
    url_text  = f'https://api.telegram.org/bot{BOT_TOKEN}/sendMessage'

    # --- Tentukan strategi caption ---
    if len(caption) <= MAX_CAPTION:
        caption_with_photo = caption   # kirim bersama foto
        overflow_text      = ""        # tidak ada sisa
    else:
        caption_with_photo = ""        # foto tanpa caption
        overflow_text      = caption   # seluruh teks jadi pesan terpisah

    # --- Kirim foto ---
    for attempt in range(retry):
        try:
            with open(image_path, 'rb') as img:
                payload = {'chat_id': chat_id, 'parse_mode': 'HTML'}
                if caption_with_photo:
                    payload['caption'] = caption_with_photo

                resp = requests.post(url_photo, data=payload,
                                     files={'photo': img}, timeout=15)

                if resp.status_code == 200:
                    break  # foto berhasil, lanjut ke overflow jika ada

                if resp.status_code == 429:
                    retry_after = resp.json().get('parameters', {}).get('retry_after', 5)
                    with output_area:
                        print(f"⏳ Rate limited. Tunggu {retry_after} detik...")
                    time.sleep(retry_after + 1)
                    continue

                err_desc = resp.json().get('description', '')
                if 'blocked' in err_desc or 'not found' in err_desc or 'deactivated' in err_desc:
                    return f'failed:{err_desc}'

                time.sleep(DELAY_RETRY)

        except requests.exceptions.Timeout:
            time.sleep(DELAY_RETRY)
        except Exception:
            time.sleep(DELAY_RETRY)
    else:
        return 'failed'  # semua retry habis, foto tidak terkirim

    # --- Kirim overflow teks jika caption terlalu panjang ---
    if overflow_text:
        time.sleep(DELAY_ANTAR_GAMBAR)
        for attempt in range(retry):
            try:
                resp = requests.post(url_text, data={
                    'chat_id': chat_id,
                    'text': overflow_text,
                    'parse_mode': 'HTML'
                }, timeout=10)

                if resp.status_code == 200:
                    return 'success'

                if resp.status_code == 429:
                    retry_after = resp.json().get('parameters', {}).get('retry_after', 5)
                    time.sleep(retry_after + 1)
                    continue

                time.sleep(DELAY_RETRY)

            except Exception:
                time.sleep(DELAY_RETRY)

        return 'photo_ok_text_failed'

    return 'success'


def load_chat_ids():
    """Baca semua chat_id dari CSV di Drive."""
    if not os.path.exists(CSV_FILE):
        with output_area:
            print(f"❌ File karyawan.csv tidak ditemukan di {DRIVE_DIR}.")
            print("   Harap upload file tersebut ke folder Drive di atas.")
        return []
    with open(CSV_FILE, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        return [row[0].strip() for row in reader if row and row[0].strip()]


def load_success_ids():
    """
    Kembalikan set chat_id yang sudah berhasil dikirim (status mengandung 'success').
    Yang GAGAL (failed) TIDAK dimasukkan → akan dilewati / diabaikan juga.
    Hanya yang belum ada di log sama sekali yang akan diproses.
    """
    success_ids = set()
    failed_ids  = set()
    if not os.path.exists(LOG_FILE):
        return success_ids, failed_ids
    with open(LOG_FILE, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 2:
                continue
            cid, status = row[0].strip(), row[1].strip()
            if 'success' in status:
                success_ids.add(cid)
            else:
                failed_ids.add(cid)
    return success_ids, failed_ids


def log_result(chat_id, status):
    """Tulis hasil pengiriman ke log di Drive."""
    with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([chat_id, status, datetime.now().isoformat()])


def show_log_summary():
    """Tampilkan ringkasan log saat ini."""
    success_ids, failed_ids = load_success_ids()
    all_ids = load_chat_ids()
    done    = success_ids | failed_ids          # sudah pernah dicoba
    pending = [i for i in all_ids if i not in done]
    status_label.value = (
        f"<b>📊 Status Log</b> — "
        f"✅ Berhasil: <b>{len(success_ids)}</b> | "
        f"⏩ Dilewati/Gagal: <b>{len(failed_ids)}</b> | "
        f"⏳ Belum dikirim: <b>{len(pending)}</b>"
    )


# ============================================================
# === MODE & UPLOAD OBSERVER =================================
# ============================================================

def on_mode_change(change):
    mode = change['new']
    text_message.layout.display   = 'block' if mode != '🖼️ Hanya Gambar + Caption' else 'none'
    upload_button.layout.display  = 'none'  if mode == '📝 Hanya Teks' else 'block'
    if mode == '📝 Hanya Teks':
        captions_container.children = []

mode_selector.observe(on_mode_change, names='value')


def update_caption_inputs(change):
    with output_area:
        clear_output(wait=True)
        if not upload_button.value:
            captions_container.children = []
            return
        print("🔄 Memuat preview gambar...")

    caption_rows = []
    for i, (filename, file_info) in enumerate(upload_button.value.items()):
        fmt = 'jpeg' if filename.lower().endswith(('.jpg','.jpeg')) else \
              'gif'  if filename.lower().endswith('.gif') else \
              'webp' if filename.lower().endswith('.webp') else 'png'
        img_widget = widgets.Image(value=file_info['content'], format=fmt, width=150, height=150)
        caption_input = widgets.Textarea(
            placeholder=f'Caption untuk "{filename}" (opsional)',
            description=f'Caption {i+1}:',
            layout=widgets.Layout(width='100%', height='80px')
        )
        row = widgets.HBox([
            widgets.VBox([img_widget, widgets.Label(f"📁 {filename}")],
                         layout=widgets.Layout(width='220px', align_items='center')),
            caption_input
        ], layout=widgets.Layout(margin='10px 0', align_items='flex-start'))
        caption_rows.append(row)
    captions_container.children = caption_rows

upload_button.observe(update_caption_inputs, names='value')


# ============================================================
# === TOMBOL KIRIM ===========================================
# ============================================================

def on_send_clicked(b):
    with output_area:
        clear_output()
        mode = mode_selector.value

        # ── Validasi ──────────────────────────────────────────
        if mode == '📝 Hanya Teks' and not text_message.value.strip():
            print("🚫 Harap isi pesan teks terlebih dahulu.")
            return
        if mode != '📝 Hanya Teks' and not upload_button.value:
            print("🚫 Harap upload gambar terlebih dahulu.")
            return

        # ── Muat data ─────────────────────────────────────────
        all_ids = load_chat_ids()
        if not all_ids:
            return

        success_ids, failed_ids = load_success_ids()

        # Yang sudah SUCCESS → skip
        # Yang GAGAL → skip juga (abaikan, sesuai permintaan)
        # Hanya yang belum ada di log sama sekali → kirim
        already_done = success_ids | failed_ids
        pending_ids  = [cid for cid in all_ids if cid not in already_done]

        print(f"📋 Total di CSV       : {len(all_ids)}")
        print(f"✅ Sudah berhasil     : {len(success_ids)}")
        print(f"⏩ Gagal (diabaikan)  : {len(failed_ids)}")
        print(f"⏳ Akan dikirim       : {len(pending_ids)}")

        if not pending_ids:
            print("\n✅ Semua user sudah diproses (berhasil/gagal diabaikan).")
            show_log_summary()
            return

        # ── Siapkan gambar ke Drive ────────────────────────────
        image_paths, captions = [], []
        if mode != '📝 Hanya Teks':
            for child in captions_container.children:
                captions.append(child.children[1].value.strip())

            for filename, file_info in upload_button.value.items():
                save_path = os.path.join(IMAGE_DIR, filename)
                with open(save_path, 'wb') as f:
                    f.write(file_info['content'])
                image_paths.append(save_path)

            print(f"🖼️  Gambar per user   : {len(image_paths)}")

        print("\n" + "─"*45)

        # ── Loop pengiriman ────────────────────────────────────
        for i, chat_id in enumerate(pending_ids):
            prefix = f"[{i+1}/{len(pending_ids)}] {chat_id}"

            if mode == '📝 Hanya Teks':
                status = send_message_to_chat(chat_id, text_message.value)
                log_result(chat_id, f'text:{status}')
                icon = '✅' if status == 'success' else '❌'
                print(f"{icon} {prefix} → {status}")
                time.sleep(DELAY_ANTAR_PESAN)

            else:
                # Kirim teks dulu jika mode campuran
                if mode == '📝🖼️ Teks + Gambar' and text_message.value.strip():
                    st = send_message_to_chat(chat_id, text_message.value)
                    log_result(chat_id, f'text:{st}')
                    time.sleep(DELAY_ANTAR_GAMBAR)

                all_ok = True
                for j, (img_path, cap) in enumerate(zip(image_paths, captions)):
                    st = send_photo_to_chat(chat_id, img_path, cap)
                    log_result(chat_id, f'image_{j+1}:{st}')
                    if st != 'success':
                        all_ok = False
                    time.sleep(DELAY_ANTAR_PESAN)

                icon = '✅' if all_ok else '⚠️'
                print(f"{icon} {prefix}")

        print("\n🎉 Sesi broadcast selesai!")
        show_log_summary()

send_button.on_click(on_send_clicked)


# ============================================================
# === TOMBOL RESET LOG =======================================
# ============================================================

def on_reset_log(b):
    with output_area:
        clear_output()
        if os.path.exists(LOG_FILE):
            os.remove(LOG_FILE)
            print("🗑️  Log dihapus. Broadcast berikutnya akan mulai dari awal.")
        else:
            print("ℹ️  File log belum ada.")
        show_log_summary()

reset_log_button.on_click(on_reset_log)


# ============================================================
# === TAMPILKAN GUI ==========================================
# ============================================================
show_log_summary()

display(widgets.VBox([
    widgets.HTML("<h3>📢 Broadcast Telegram — Persistent via Google Drive</h3>"),
    status_label,
    widgets.HTML("<hr>"),
    mode_selector,
    text_message,
    upload_button,
    widgets.Label("✍️ Isi caption untuk setiap gambar (opsional):"),
    captions_container,
    widgets.HBox([send_button, reset_log_button],
                 layout=widgets.Layout(gap='12px', margin='8px 0')),
    output_area
]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive terhubung.
📁 Folder kerja : /content/drive/MyDrive/TelegramBroadcast
📋 CSV karyawan : /content/drive/MyDrive/TelegramBroadcast/id_telegram_test.csv
📝 File log     : /content/drive/MyDrive/TelegramBroadcast/broadcast_log.csv
🖼️  Folder gambar: /content/drive/MyDrive/TelegramBroadcast/images
